# Notebook setup

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/LC-Movie-Genre-Prediction/notebooks
/home/turbotowerlnx/Documents/Master/LC-Movie-Genre-Prediction/app


# Training loss

In [7]:
import pandas as pd
import plotly.express as px
path_metrics = "results/metrics.csv"

# Read metrics file
pd_metrics = pd.read_csv(path_metrics)

# Quick inspect (prints column names) -- can be removed later
print('metrics columns:', pd_metrics.columns.tolist())

# Ensure eval_f1 exists and drop NaNs for plotting

pd_plot = pd_metrics[['eval_f1']].copy()
pd_plot['eval_f1'] = pd_plot['eval_f1'].astype(float)

# Choose x-axis: prefer 'epoch', then 'step', otherwise use index
if 'epoch' in pd_metrics.columns:
    pd_plot['x'] = pd_metrics['epoch']
    x_label = 'Epoch'
elif 'step' in pd_metrics.columns:
    pd_plot['x'] = pd_metrics['step']
    x_label = 'Step'
else:
    pd_plot = pd_plot.reset_index().rename(columns={'index': 'x'})
    x_label = 'Index'

# Drop rows where eval_f1 is NaN
pd_plot = pd_plot.dropna(subset=['eval_f1'])

# Build the Plotly line chart
fig = px.line(pd_plot, x='x', y='eval_f1', markers=True,
              title='Evaluation F1 over ' + x_label,
              labels={'eval_f1': 'Eval F1', 'x': x_label})

# Set a sensible y-range padding (works with decimals or percentages)
yvals = pd_plot['eval_f1']
if not yvals.empty:
    y_min = float(yvals.min())
    y_max = float(yvals.max())
    pad = (y_max - y_min) * 0.05 if y_max != y_min else 0.05
    fig.update_layout(yaxis=dict(range=[y_min - pad, y_max + pad]))

fig.update_traces(line=dict(width=2))
fig.show()

metrics columns: ['loss', 'grad_norm', 'learning_rate', 'epoch', 'step', 'eval_loss', 'eval_accuracy', 'eval_f1', 'eval_precision', 'eval_recall', 'eval_hamming_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'train_runtime', 'train_samples_per_second', 'train_steps_per_second', 'total_flos', 'train_loss']


# Results

In [9]:
import plotly.express as px
def plot_model_performance(models_list, label):
    models_df = pd.DataFrame(models_list, columns=["model_name", "f1_score"])
    models_df = models_df.sort_values(by="f1_score", ascending=True).reset_index(drop=True)

    fig = px.bar(
        models_df,
        x="model_name",
        y="f1_score",
        color="model_name",
        labels={"f1_score": "F1 Score (%)", "model_name": "Model"},
        hover_data={"f1_score": True, "model_name": False},
        title=label,
        text="f1_score"
    )

    fig.update_traces(texttemplate='%{text}', textposition='outside')

    fig.update_layout(
        legend_title_text="Model",
        yaxis=dict(range=[models_df["f1_score"].min() - 10, models_df["f1_score"].max() + 10]),
        xaxis=dict(showticklabels=False, title=""),
        width=900,
        height=520,
        margin=dict(l=40, r=40, t=80, b=120)
    )
    fig.show()

In [10]:
models = [
    ("bert-base", 63.5),
    ("roberta-base", 63.5),
    ("deberta-v3", 64),
    ("xlm-roberta-base", 62),
    ("roberta-large-mnli", 66.7)
]


plot_model_performance(models, "F1 Transformer Models")

In [15]:
models = [
    ("Logistic Regression", 57.3),
    ("Random Forest", 48.2),
    ("XGBoost", 51.8),
    ("SVC", 45.6),
]


plot_model_performance(models, "F1 Classic Models")

In [12]:
models = [
    ("Gemini", 83.4)
]


plot_model_performance(models, "F1 LLMs")

In [13]:
models = [
    ("Logistic Regression", 57.2),
    ("roberta-large-mnli", 66.7),
    ("Gemini", 83.4),
]


plot_model_performance(models, "Best models")